# Joe — Google Colab API

Este notebook transforma uma sessão do Google Colab em um backend remoto temporário para o aplicativo desktop Joe.

## O que ele inicia
- **Texto → Áudio:** Kyutai Pocket TTS (CPU).
- **Áudio → Texto:** Moondream Parakeet Redux via Photon (usa CUDA quando uma GPU está disponível).
- **API:** FastAPI protegida por token Bearer.
- **Acesso externo:** Cloudflare Tunnel.

## Configuração persistente (faça uma vez)

Para não precisar copiar uma nova URL e um novo token a cada sessão, use **Colab Secrets** com um domínio fixo do Cloudflare:

1. No Cloudflare, crie um **Named Tunnel** e adicione uma rota pública para seu subdomínio, apontando para `http://localhost:8000`.
2. No Colab, abra o painel **Secrets** (ícone de chave à esquerda) e crie estes três Secrets:
   - `JOE_API_URL`: URL HTTPS fixa, por exemplo `https://api.seudominio.com`.
   - `JOE_API_TOKEN`: token Bearer forte e permanente, usado pelo aplicativo Joe.
   - `CLOUDFLARE_TUNNEL_TOKEN`: token do Named Tunnel criado no Cloudflare.
3. Ative **Notebook access** para os três Secrets.
4. Execute as células em sequência. Com `JOE_API_TOKEN` configurado, o notebook carregará as credenciais automaticamente e não exibirá o token Bearer.
5. Na primeira configuração, copie `JOE_API_URL` e `JOE_API_TOKEN` do painel Secrets para a aba **Google Colab / API** do Joe. O desktop salva esses campos localmente; não será necessário digitá-los novamente enquanto os Secrets e o domínio forem mantidos.

**Importante:** `JOE_API_TOKEN` é a senha da API do Joe; `CLOUDFLARE_TUNNEL_TOKEN` autentica o túnel. São credenciais diferentes. Não publique nem compartilhe seus valores.

### Alternativa sem domínio próprio

O modo **Quick Tunnel** continua disponível para testes. Nesse modo, o Cloudflare gera uma URL `trycloudflare.com` aleatória a cada sessão. Portanto, a URL precisa ser atualizada no Joe quando o runtime reiniciar, mesmo que o token Bearer esteja salvo.

O Colab não garante disponibilidade ou duração da sessão. O runtime e o túnel podem ser encerrados por inatividade ou pelos limites do serviço. <https://research.google.com/colaboratory/faq.html>


In [ ]:
#@title Baixar o código do Joe
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/joe')
REPO_URL = 'https://github.com/g4b3r-desktop/joe.git'
BRANCH = 'feature/joe-studio-improvements'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    # Atualiza o clone quando o notebook é reutilizado na mesma sessão do Colab.
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('Código do Joe pronto:', REPO_DIR)


In [ ]:
#@title Instalar dependências e verificar GPU
%pip install -q -r /content/joe/colab/requirements.txt

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Aviso: sem GPU disponível; o Parakeet será executado na CPU.')


## Monitorar CPU, RAM, GPU e VRAM

Execute a célula abaixo para acompanhar os recursos do ambiente Colab a cada 2 segundos. Ela permanece ativa enquanto atualiza os valores; use o botão **Parar [■]** da própria célula para interromper o monitoramento e liberar a execução do notebook.


In [ ]:
#@title Monitorar CPU, RAM, GPU e VRAM
%pip install -q psutil gputil

import psutil
import time
from IPython.display import clear_output
import GPUtil

def monitor_resources_loop(interval_seconds=2):
    try:
        while True:
            clear_output(wait=True)
            print("=== Monitoramento do Sistema (use o botão Parar [■] para interromper) ===\n")

            # CPU
            cpu_usage = psutil.cpu_percent()
            cpu_cores = psutil.cpu_count()
            print(f"CPU: {cpu_usage}% (Total de Cores: {cpu_cores})")

            # RAM
            ram = psutil.virtual_memory()
            ram_total = ram.total / (1024**3)
            ram_used = ram.used / (1024**3)
            print(f"RAM: {ram_used:.2f} GB / {ram_total:.2f} GB ({ram.percent}% em uso)")

            # GPU & VRAM
            gpu_query_ok = True
            try:
                gpus = GPUtil.getGPUs()
            except Exception as exc:
                gpus = []
                gpu_query_ok = False
                print(f"\nGPU/VRAM: não foi possível consultar a GPU ({exc})")

            if gpus:
                for i, gpu in enumerate(gpus):
                    print(f"\n=== GPU {i}: {gpu.name} ===")
                    print(f"Uso da GPU: {gpu.load*100:.1f}%")
                    print(f"VRAM (Memória de Vídeo): {gpu.memoryUsed:.0f} MB / {gpu.memoryTotal:.0f} MB ({gpu.memoryUtil*100:.1f}% em uso)")
                    print(f"Temperatura: {gpu.temperature:.0f}°C")
            elif gpu_query_ok:
                print("\nGPU/VRAM: nenhuma GPU ativa ou disponível neste ambiente de execução.")

            time.sleep(interval_seconds)
    except KeyboardInterrupt:
        print("\nMonitoramento interrompido pelo usuário.")

# Inicia o monitoramento atualizando a cada 2 segundos
monitor_resources_loop(interval_seconds=2)


## Google Drive: carregar modelos e escolher pastas

O Pocket TTS deve usar os arquivos locais extraídos de `Meu Drive/3d_models_ia/JOE_MODELS.zip`. A API não inicia em modo Drive se os pesos e o tokenizer não forem encontrados. O ZIP é extraído para `/content/joe/models/` e reutilizado durante a mesma sessão Colab.

Estrutura esperada (pode estar diretamente na raiz ou dentro de `JOE_MODELS/`):

```text
pocket-tts/
└── languages/portuguese/model.safetensors
pocket-tts-without-voice-cloning/
└── languages/portuguese/
    ├── model.safetensors        # opcional, fallback sem clonagem
    └── tokenizer.json
```

Para síntese com clonagem, o arquivo `pocket-tts/languages/portuguese/model.safetensors` precisa ser o checkpoint **com suporte a voice cloning**. Se o Joe não enviar um áudio de referência na solicitação, informe um WAV local no campo **Voz padrão WAV**. Não é usado um WAV remoto `hf://` como fallback.


In [ ]:
#@title Montar o Google Drive e configurar os modelos de voz
import os
import zipfile
from pathlib import Path

import ipywidgets as widgets
import yaml
from google.colab import drive
from IPython.display import display, clear_output

drive.mount('/content/drive')

DEFAULT_ZIP_PATH = Path('/content/drive/MyDrive/3d_models_ia/modelos-voz/JOE_MODELS.zip')
EXTRACT_BASE = Path('/content/joe/models')

zip_widget = widgets.Text(
    value=str(DEFAULT_ZIP_PATH),
    description='Arquivo ZIP:',
    layout=widgets.Layout(width='95%'),
)
model_folder_widget = widgets.Text(
    value=str(EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts'),
    description='Pasta Pocket TTS:',
    layout=widgets.Layout(width='95%'),
)
tokenizer_widget = widgets.Text(
    value=str(EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts-without-voice-cloning' / 'languages' / 'portuguese' / 'tokenizer.json'),
    description='Tokenizer JSON:',
    layout=widgets.Layout(width='95%'),
)
voice_widget = widgets.Text(
    value='',
    placeholder='Obrigatório se o Joe não enviar um WAV: caminho de um WAV no Drive',
    description='Voz padrão WAV:',
    layout=widgets.Layout(width='95%'),
)
hf_cache_widget = widgets.Text(
    value=str(Path('/content/drive/MyDrive/3d_models_ia/huggingface-cache')),
    description='Cache ASR:',
    layout=widgets.Layout(width='95%'),
)
use_drive_tts_widget = widgets.Checkbox(
    value=True,
    description='Obrigatório: carregar Pocket TTS do Google Drive',
)
apply_paths_button = widgets.Button(
    description='Aplicar pastas e modelos',
    button_style='success',
    icon='check',
)
paths_output = widgets.Output()


def _extract_models_zip(zip_path: Path) -> Path:
    if not zip_path.is_file():
        raise FileNotFoundError(f'Arquivo ZIP não encontrado: {zip_path}')

    # Reutiliza extração válida na mesma sessão do Colab.
    for candidate in (EXTRACT_BASE / 'JOE_MODELS', EXTRACT_BASE):
        if (
            (candidate / 'pocket-tts' / 'languages' / 'portuguese' / 'model.safetensors').is_file()
            and any(p.name == 'tokenizer.json' for p in candidate.rglob('tokenizer.json'))
        ):
            return candidate

    EXTRACT_BASE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        base_resolved = EXTRACT_BASE.resolve()
        for member in archive.infolist():
            target = (EXTRACT_BASE / member.filename).resolve()
            if not target.is_relative_to(base_resolved):
                raise ValueError(f'Caminho inseguro dentro do ZIP: {member.filename}')
        archive.extractall(EXTRACT_BASE)

    for candidate in (EXTRACT_BASE / 'JOE_MODELS', EXTRACT_BASE):
        if (candidate / 'pocket-tts').is_dir():
            return candidate
    raise FileNotFoundError(
        'Não encontrei pocket-tts/ dentro do ZIP. Confira a estrutura do JOE_MODELS.zip.'
    )


def apply_drive_model_paths(_=None):
    with paths_output:
        clear_output()
        for key in ('JOE_TTS_CONFIG_PATH', 'JOE_TTS_DEFAULT_VOICE', 'JOE_TTS_USE_DRIVE_MODELS'):
            os.environ.pop(key, None)

        try:
            if not use_drive_tts_widget.value:
                raise RuntimeError(
                    'Este notebook foi configurado para usar os modelos do Drive. '
                    'Marque a opção de carregamento pelo Drive.'
                )

            zip_path = Path(zip_widget.value.strip()).expanduser()
            extracted_root = _extract_models_zip(zip_path)
            model_root = Path(model_folder_widget.value.strip()).expanduser()
            if not model_root.is_dir() or model_root == EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts':
                model_root = extracted_root / 'pocket-tts'
                model_folder_widget.value = str(model_root)

            clone_weights = model_root / 'languages' / 'portuguese' / 'model.safetensors'
            no_clone_weights = (
                extracted_root / 'pocket-tts-without-voice-cloning'
                / 'languages' / 'portuguese' / 'model.safetensors'
            )
            # O ZIP pode conter pastas extras ou nomes de diretório diferentes.
            # Primeiro respeita o caminho informado; depois procura na árvore extraída.
            tokenizer_file = Path(tokenizer_widget.value.strip()).expanduser()
            if not tokenizer_file.is_file():
                candidates = list(extracted_root.rglob('tokenizer.json'))
                portuguese_candidates = [
                    p for p in candidates if 'portuguese' in str(p).lower()
                ]
                preferred_candidates = [
                    p for p in portuguese_candidates
                    if 'without-voice-cloning' in str(p).lower()
                ]
                search_candidates = preferred_candidates or portuguese_candidates or candidates
                if len(search_candidates) == 1:
                    tokenizer_file = search_candidates[0]
                elif len(search_candidates) > 1:
                    raise FileNotFoundError(
                        'Encontrei vários tokenizer.json. Informe o caminho correto no campo Tokenizer JSON:\\n'
                        + '\\n'.join(str(p) for p in search_candidates)
                    )
                else:
                    raise FileNotFoundError(
                        'Não encontrei tokenizer.json dentro do ZIP extraído. '
                        'Confira se o arquivo foi incluído no JOE_MODELS.zip.'
                    )
                tokenizer_widget.value = str(tokenizer_file)

            missing = [str(p) for p in (clone_weights, tokenizer_file) if not p.is_file()]
            if missing:
                raise FileNotFoundError(
                    'Faltam arquivos obrigatórios no ZIP do Drive:\\n' + '\\n'.join(missing)
                )

            import pocket_tts
            package_config = Path(pocket_tts.__file__).resolve().parent / 'config' / 'portuguese.yaml'
            if not package_config.is_file():
                raise FileNotFoundError(f'Configuração base não encontrada: {package_config}')

            config_data = yaml.safe_load(package_config.read_text(encoding='utf-8'))
            config_data['weights_path'] = str(clone_weights.resolve())
            config_data['weights_path_without_voice_cloning'] = (
                str(no_clone_weights.resolve()) if no_clone_weights.is_file() else None
            )
            config_data['flow_lm']['lookup_table']['tokenizer_path'] = str(tokenizer_file.resolve())

            voice_value = voice_widget.value.strip()
            if voice_value:
                voice_file = Path(voice_value).expanduser()
                if not voice_file.is_file():
                    raise FileNotFoundError(f'WAV de voz padrão não encontrado: {voice_file}')
                if voice_file.suffix.lower() != '.wav':
                    raise ValueError('A voz padrão deve ser um arquivo WAV de referência.')
            else:
                voice_file = None

            runtime_dir = Path('/content/joe/runtime')
            runtime_dir.mkdir(parents=True, exist_ok=True)
            local_config = runtime_dir / 'portuguese_from_drive.yaml'
            local_config.write_text(
                yaml.safe_dump(config_data, sort_keys=False, allow_unicode=True),
                encoding='utf-8',
            )

            # A API só inicia em modo Drive depois que a configuração foi validada.
            os.environ['JOE_TTS_CONFIG_PATH'] = str(local_config)
            os.environ['JOE_TTS_USE_DRIVE_MODELS'] = '1'
            if voice_file:
                os.environ['JOE_TTS_DEFAULT_VOICE'] = str(voice_file.resolve())

            cache_dir = Path(hf_cache_widget.value.strip()).expanduser()
            cache_dir.mkdir(parents=True, exist_ok=True)
            os.environ['HF_HOME'] = str(cache_dir)

            print(f'ZIP extraído em: {extracted_root}')
            print(f'Pesos com clonagem (Drive): {clone_weights}')
            print(
                'Pesos sem clonagem (Drive): '
                + (str(no_clone_weights) if no_clone_weights.is_file() else 'não incluídos')
            )
            print(f'Tokenizer (Drive): {tokenizer_file}')
            print(f'WAV padrão: {voice_file if voice_file else "não definido; o Joe deverá enviar um WAV"}')
            print(f'Configuração local: {local_config}')
            print('Configuração validada: a API usará os pesos locais do Drive.')
        except Exception as exc:
            print(f'ERRO ao configurar os modelos do Drive: {exc}')
            print('A API não deve ser iniciada até corrigir os caminhos/arquivos do ZIP.')


apply_paths_button.on_click(apply_drive_model_paths)
display(widgets.VBox([
    use_drive_tts_widget,
    zip_widget,
    model_folder_widget,
    tokenizer_widget,
    voice_widget,
    hf_cache_widget,
    apply_paths_button,
    paths_output,
]))
apply_drive_model_paths()


## Escolher o endereço do túnel Cloudflare

Quando os três Secrets `JOE_API_URL`, `JOE_API_TOKEN` e `CLOUDFLARE_TUNNEL_TOKEN` estiverem configurados e com **Notebook access** habilitado, o modo **domínio fixo** será selecionado automaticamente. A URL pode ser informada como `api.seudominio.com` ou `https://api.seudominio.com`. Os campos abaixo também permitem configurar um Named Tunnel apenas para esta sessão.


In [ ]:
#@title Carregar Secrets e configurar o Cloudflare Tunnel
import os
import re
from urllib.parse import urlsplit

import ipywidgets as widgets
from google.colab import userdata
from IPython.display import display


def _read_colab_secret(name):
    try:
        value = userdata.get(name)
        return str(value).strip() if value is not None else ""
    except Exception:
        # Secret ausente ou Notebook access ainda não habilitado.
        return ""


def _domain_from_api_url(value):
    """Retorna apenas um hostname HTTPS válido, aceitando URL ou domínio puro."""
    raw_value = str(value or "").strip()
    if not raw_value or re.search(r"\s", raw_value):
        return ""

    # O Secret pode conter api.exemplo.com ou https://api.exemplo.com.
    url_to_parse = raw_value
    if not url_to_parse.lower().startswith(("http://", "https://")):
        url_to_parse = "https://" + url_to_parse

    try:
        parsed = urlsplit(url_to_parse)
        hostname = parsed.hostname
        port = parsed.port
    except ValueError:
        return ""

    if (
        parsed.scheme.lower() != "https"
        or not hostname
        or parsed.username is not None
        or parsed.password is not None
        or parsed.path not in ("", "/")
        or parsed.query
        or parsed.fragment
        or port is not None
    ):
        return ""

    hostname = hostname.lower().rstrip(".")
    domain_pattern = r"(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}"
    if not re.fullmatch(domain_pattern, hostname):
        return ""
    return hostname


# Os Secrets ficam guardados no painel do Colab e são carregados no ambiente
# desta sessão. Não coloque os valores diretamente no notebook.
SAVED_API_URL = _read_colab_secret("JOE_API_URL")
if SAVED_API_URL and not SAVED_API_URL.lower().startswith(("http://", "https://")):
    SAVED_API_URL = "https://" + SAVED_API_URL

SAVED_API_TOKEN = _read_colab_secret("JOE_API_TOKEN")
SAVED_TUNNEL_TOKEN = _read_colab_secret("CLOUDFLARE_TUNNEL_TOKEN")

SAVED_TUNNEL_DOMAIN = _domain_from_api_url(SAVED_API_URL)
if SAVED_API_TOKEN:
    os.environ["JOE_API_TOKEN"] = SAVED_API_TOKEN
else:
    os.environ.pop("JOE_API_TOKEN", None)

# Seleciona o Named Tunnel automaticamente somente quando os três Secrets
# necessários para uma conexão persistente estiverem disponíveis.
if SAVED_TUNNEL_DOMAIN and SAVED_API_TOKEN and SAVED_TUNNEL_TOKEN:
    os.environ["JOE_API_URL"] = SAVED_API_URL.rstrip("/")
    os.environ["CLOUDFLARE_TUNNEL_TOKEN"] = SAVED_TUNNEL_TOKEN
    JOE_TUNNEL_MODE = "custom"
    JOE_TUNNEL_DOMAIN = SAVED_TUNNEL_DOMAIN
    JOE_TUNNEL_TOKEN = SAVED_TUNNEL_TOKEN
else:
    os.environ.pop("JOE_API_URL", None)
    os.environ.pop("CLOUDFLARE_TUNNEL_TOKEN", None)
    JOE_TUNNEL_MODE = "quick"
    JOE_TUNNEL_DOMAIN = ""
    JOE_TUNNEL_TOKEN = ""

missing_secrets = [
    name
    for name, value in (
        ("JOE_API_URL", SAVED_TUNNEL_DOMAIN),
        ("JOE_API_TOKEN", SAVED_API_TOKEN),
        ("CLOUDFLARE_TUNNEL_TOKEN", SAVED_TUNNEL_TOKEN),
    )
    if not value
]
if not missing_secrets:
    print("Secrets persistentes carregados. O notebook usará o domínio fixo e o mesmo token Bearer.")
else:
    print("Secrets ainda não configurados ou sem acesso: " + ", ".join(missing_secrets))
    print("Abra o painel Secrets (ícone de chave), confirme os nomes e habilite Notebook access.")
    print("A conexão persistente exige os três Secrets. O Quick Tunnel continua disponível como alternativa temporária.")

tunnel_mode_widget = widgets.Dropdown(
    options=[
        ("URL temporária (trycloudflare.com)", "quick"),
        ("Meu domínio Cloudflare (Named Tunnel)", "custom"),
    ],
    value=JOE_TUNNEL_MODE,
    description="Tipo de túnel:",
    layout=widgets.Layout(width="95%"),
)
tunnel_domain_widget = widgets.Text(
    value=JOE_TUNNEL_DOMAIN,
    placeholder="api.seudominio.com",
    description="Domínio:",
    layout=widgets.Layout(width="95%"),
)
tunnel_token_widget = widgets.Password(
    value=JOE_TUNNEL_TOKEN,
    placeholder="Token do Cloudflare Named Tunnel",
    description="Token túnel:",
    layout=widgets.Layout(width="95%"),
)
tunnel_config_button = widgets.Button(
    description="Aplicar configuração do túnel",
    button_style="success",
    icon="check",
)
tunnel_config_output = widgets.Output()


def apply_tunnel_config(_):
    global JOE_TUNNEL_MODE, JOE_TUNNEL_DOMAIN, JOE_TUNNEL_TOKEN

    with tunnel_config_output:
        tunnel_config_output.clear_output()
        mode = tunnel_mode_widget.value
        domain = tunnel_domain_widget.value.strip().lower().rstrip(".")
        token = tunnel_token_widget.value.strip()

        if mode == "custom":
            domain_pattern = r"(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}"
            if not re.fullmatch(domain_pattern, domain):
                print("Informe somente um domínio válido, como api.seudominio.com.")
                return
            if not token:
                print("Cole o token do Cloudflare Named Tunnel. Ele não será exibido.")
                return

            JOE_TUNNEL_MODE = "custom"
            JOE_TUNNEL_DOMAIN = domain
            JOE_TUNNEL_TOKEN = token
            os.environ["JOE_API_URL"] = f"https://{domain}"
            os.environ["CLOUDFLARE_TUNNEL_TOKEN"] = token
            print("Named Tunnel configurado para esta sessão:", os.environ["JOE_API_URL"])
            print(
                "Para persistir a conexão entre sessões, salve JOE_API_URL, "
                "JOE_API_TOKEN e CLOUDFLARE_TUNNEL_TOKEN no painel Secrets."
            )
        else:
            JOE_TUNNEL_MODE = "quick"
            JOE_TUNNEL_DOMAIN = ""
            JOE_TUNNEL_TOKEN = ""
            os.environ.pop("JOE_API_URL", None)
            os.environ.pop("CLOUDFLARE_TUNNEL_TOKEN", None)
            print("Quick Tunnel temporário selecionado. A URL mudará quando uma nova sessão for iniciada.")

        print("Agora execute a célula “Iniciar a API e o túnel”.")


tunnel_config_button.on_click(apply_tunnel_config)
display(
    widgets.VBox(
        [
            tunnel_mode_widget,
            tunnel_domain_widget,
            tunnel_token_widget,
            tunnel_config_button,
            tunnel_config_output,
        ]
    )
)


## Iniciar a API e o túnel

Depois de aplicar os caminhos dos modelos do Google Drive e conferir a configuração do túnel, execute a célula de código abaixo. Ela inicia a API FastAPI, autentica com `JOE_API_TOKEN` e publica o serviço pelo Cloudflare Tunnel.

Com os três Secrets configurados, a inicialização reutiliza o mesmo domínio e as mesmas credenciais. Não é necessário copiar novamente a URL ou o token Bearer para o Joe.


In [ ]:
#@title Iniciar a API do Joe e publicar o túnel
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
import queue
import requests
from pathlib import Path
from IPython.display import display, Markdown

# Impede iniciar a API se a configuração local do Drive não foi aplicada.
if os.environ.get('JOE_TTS_USE_DRIVE_MODELS') != '1':
    raise RuntimeError(
        'Modelos do Pocket TTS do Google Drive não foram configurados. '
        'Volte à célula Google Drive, corrija os arquivos e clique em Aplicar.'
    )
if not os.environ.get('JOE_TTS_CONFIG_PATH') or not Path(os.environ['JOE_TTS_CONFIG_PATH']).is_file():
    raise RuntimeError('Arquivo YAML local do Pocket TTS não encontrado. Reaplique a configuração do Drive.')

# Reutiliza o token persistente do Colab Secrets. Só gera um token novo
# quando JOE_API_TOKEN ainda não foi configurado.
API_TOKEN = os.environ.get('JOE_API_TOKEN', '').strip()
GENERATED_API_TOKEN = not bool(API_TOKEN)
if GENERATED_API_TOKEN:
    API_TOKEN = secrets.token_urlsafe(36)
os.environ['JOE_API_TOKEN'] = API_TOKEN

# Instala o binário oficial cloudflared para criar o túnel.
cloudflared_path = shutil.which('cloudflared')
if cloudflared_path is None:
    binary = '/usr/local/bin/cloudflared'
    subprocess.run(
        ['wget', '-q', '-O', binary,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        check=True,
    )
    os.chmod(binary, 0o755)
    cloudflared_path = binary

# Libera a porta 8000 antes de iniciar uma nova instância da API.
subprocess.run(
    ['bash', '-c', 'fuser -k 8000/tcp'],
    capture_output=True,
    text=True,
)
time.sleep(2)
print('Porta 8000 liberada.')

server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api_server:app',
     '--host', '127.0.0.1', '--port', '8000', '--no-access-log'],
    cwd='/content/joe/colab',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Confirma que a API iniciou antes de publicar o túnel.
api_ready = False
for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError('O servidor Joe API terminou inesperadamente.')
    try:
        response = requests.get(
            'http://127.0.0.1:8000/health',
            headers={'Authorization': f'Bearer {API_TOKEN}'},
            timeout=2,
        )
        if response.status_code == 200:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not api_ready:
    raise RuntimeError('A API não respondeu. Confira a saída do processo uvicorn.')

JOE_TUNNEL_MODE = globals().get('JOE_TUNNEL_MODE', 'quick')
JOE_TUNNEL_DOMAIN = globals().get('JOE_TUNNEL_DOMAIN', '')
JOE_TUNNEL_TOKEN = globals().get('JOE_TUNNEL_TOKEN', '')

tunnel_lines = queue.Queue()
tunnel_env = os.environ.copy()
if JOE_TUNNEL_MODE == 'custom':
    if not JOE_TUNNEL_DOMAIN or not JOE_TUNNEL_TOKEN:
        raise RuntimeError('Configure JOE_API_URL e CLOUDFLARE_TUNNEL_TOKEN nos Secrets ou na célula de configuração do túnel.')
    API_URL = os.environ.get('JOE_API_URL', f'https://{JOE_TUNNEL_DOMAIN}').strip().rstrip('/')
    if _domain_from_api_url(API_URL) != JOE_TUNNEL_DOMAIN:
        raise RuntimeError('JOE_API_URL precisa corresponder ao domínio configurado no Named Tunnel.')
    # TUNNEL_TOKEN evita expor o segredo nos argumentos do processo.
    tunnel_env['TUNNEL_TOKEN'] = JOE_TUNNEL_TOKEN
    tunnel_command = [cloudflared_path, 'tunnel', '--no-autoupdate', '--loglevel', 'info', 'run']
else:
    tunnel_command = [
        cloudflared_path, 'tunnel', '--url',
        'http://127.0.0.1:8000', '--no-autoupdate'
    ]
    API_URL = None

tunnel_process = subprocess.Popen(
    tunnel_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=tunnel_env,
)

def _read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.put(line.rstrip())

threading.Thread(target=_read_tunnel_output, daemon=True).start()

if JOE_TUNNEL_MODE == 'quick':
    deadline = time.time() + 90
    while time.time() < deadline and API_URL is None:
        if tunnel_process.poll() is not None:
            raise RuntimeError('O cloudflared terminou inesperadamente.')
        try:
            line = tunnel_lines.get(timeout=1)
        except queue.Empty:
            continue
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            API_URL = match.group(0)
    if not API_URL:
        raise RuntimeError('Não foi possível obter a URL trycloudflare. Veja os logs do cloudflared.')
else:
    time.sleep(5)
    if tunnel_process.poll() is not None:
        recent_logs = []
        while not tunnel_lines.empty():
            recent_logs.append(tunnel_lines.get_nowait())
        safe_logs = '\\n'.join(recent_logs[-20:])
        raise RuntimeError(
            'O Cloudflare Named Tunnel encerrou durante a inicialização. '
            'Veja as últimas mensagens do cloudflared abaixo; o token foi mantido oculto.\\n\\n'
            + (safe_logs or 'O processo encerrou sem produzir mensagens. Confira se CLOUDFLARE_TUNNEL_TOKEN é o token de instalação do Named Tunnel.')
        )
    startup_logs = []
    while not tunnel_lines.empty():
        startup_logs.append(tunnel_lines.get_nowait())
    if startup_logs:
        print('Cloudflare:', startup_logs[-1])

# Disponibiliza a URL também como variável de ambiente durante esta sessão.
os.environ['JOE_API_URL'] = API_URL

display(Markdown(
    '## Joe API pronta\n\n'
    f'**URL da API:** `{API_URL}`\n\n'
    + (
        '**Token Bearer:** carregado de Colab Secrets (`JOE_API_TOKEN`). '
        'Use esse valor na aba Google Colab / API do Joe apenas na primeira configuração.\n\n'
        if not GENERATED_API_TOKEN else
        f'**Token Bearer:** `{API_TOKEN}`\n\n'
        '**Atenção:** este token foi gerado apenas para esta sessão. Para mantê-lo, '
        'adicione-o ao Secret `JOE_API_TOKEN` antes da próxima sessão.\n\n'
    )
    + (
        '**Conexão persistente:** o domínio e o token Bearer foram carregados dos Secrets. '
        'O Joe pode manter a mesma configuração entre sessões.\n\n'
        if JOE_TUNNEL_MODE == 'custom' and not GENERATED_API_TOKEN else
        '**Conexão temporária:** o Quick Tunnel gera outra URL quando o runtime reiniciar. '
        'Atualize a URL no Joe após cada nova sessão.\n\n'
        if JOE_TUNNEL_MODE == 'quick' else
        '**Atenção:** a URL é fixa, mas salve o token Bearer no Secret `JOE_API_TOKEN` '
        'para que ele também permaneça igual nas próximas sessões.\n\n'
    )
    + 'Não compartilhe a saída desta célula.'
))


## Encerrar a sessão

Execute esta célula quando terminar. Ela encerra o servidor e o túnel da sessão atual, mas **não apaga os Secrets** guardados no painel do Google Colab.

Com Named Tunnel + `JOE_API_URL` + `JOE_API_TOKEN` + `CLOUDFLARE_TUNNEL_TOKEN`, os mesmos valores serão reutilizados na próxima execução. Com Quick Tunnel, a URL será diferente na próxima sessão.


In [ ]:
#@title Encerrar a API e o túnel
for process_name in ('tunnel_process', 'server_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
print('Joe API e túnel encerrados.')
